# 06 — Forecast Evaluation

Metrics, horizons, segments, error attribution and uncertainty.

**MAPE is not reported.** With ~59% of observations zero it is undefined on most rows. WAPE is the headline; MASE is reported for scale-free comparison, with the caveat that the `zero` baseline wins on it.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

panel: 1,003,600 rows, 600 series, 1969 days


In [2]:
from src.run import load_results
r = load_results(cfg)
list(r)

['audit',
 'baselines',
 'train',
 'classical',
 'inventory',
 'scenarios',
 'example_forecast',
 'eda']

## Validation and held-out test

In [3]:
import pandas as pd
pd.DataFrame(r['train']['validation_aggregate']).round(4)

,model,folds,mae,rmse,wape,wape_std,mase,smape,bias,skill_vs_seasonal_naive_pct
0,lightgbm,4,1.0311,2.2453,0.7397,0.0480,0.9489,142.0785,-0.0368,14.6596
1,random_forest,4,1.0417,2.2568,0.7473,0.0353,0.9597,141.3462,-0.0389,13.7817
2,moving_average_28,4,1.0455,2.2941,0.7503,0.0402,0.9492,132.0348,-0.0403,13.4363
3,moving_average_7,4,1.0536,2.2890,0.7558,0.0323,0.9582,115.4773,-0.0653,12.8049
4,sba,4,1.1001,2.3672,0.7901,0.0606,0.9796,142.9735,-0.0110,8.8489
5,croston,4,1.1150,2.3796,0.8008,0.0617,0.9941,142.2168,0.0619,7.6130
6,seasonal_naive,4,1.2083,2.7940,0.8668,0.0396,1.0670,80.1448,-0.0653,0.0000
7,zero,4,1.3958,3.8260,1.0000,0.0000,0.9420,87.8750,-1.3958,-15.3725
8,naive,4,1.4090,3.1630,1.0105,0.0817,1.1974,83.0502,0.3421,-16.5782


In [4]:
pd.DataFrame(r['train']['test_table']).round(4)

,model,n,mae,rmse,wape,mase,smape,bias,bias_pct,zero_share_actual,pred_zero_share,skill_vs_seasonal_naive_pct
0,random_forest,16800.0,1.1384,2.6317,0.7726,1.0291,139.9733,-0.0406,-0.0276,0.556,0.3959,16.6655
1,lightgbm,16800.0,1.1443,2.6182,0.7766,1.0237,140.9095,-0.0619,-0.0420,0.556,0.4172,16.2354
2,sba,16800.0,1.1583,2.7379,0.7861,1.0017,142.0205,-0.0174,-0.0118,0.556,0.4333,15.2126
3,moving_average_28,16800.0,1.1596,2.7479,0.7870,1.0185,136.7497,0.0017,0.0011,0.556,0.4100,15.1124
4,croston,16800.0,1.1728,2.7673,0.7960,1.0163,141.2174,0.0592,0.0402,0.556,0.4183,14.1467
5,moving_average_7,16800.0,1.2055,2.8828,0.8181,1.0594,123.3934,0.0327,0.0222,0.556,0.4200,11.7572
6,seasonal_naive,16800.0,1.3661,3.3294,0.9271,1.1870,85.7760,0.0327,0.0222,0.556,0.5369,0.0000
7,zero,16800.0,1.4735,4.1261,1.0000,0.9717,88.7976,-1.4735,-1.0000,0.556,1.0000,-7.8606
8,naive,16800.0,1.6707,5.1312,1.1339,1.3668,87.8409,0.6315,0.4286,0.556,0.4667,-22.3007


## Accuracy by horizon

In [5]:
hb = pd.DataFrame(r['train']['horizon_breakdown'])
hb.pivot(index='horizon', columns='model', values='wape').round(4)

model,croston,lightgbm,moving_average_28,moving_average_7,naive,random_forest,sba,seasonal_naive,zero
horizon,,,,,,,,,
1,0.7047,0.6512,0.6808,0.7017,1.0751,0.6572,0.6924,0.8486,1.0
7,0.7512,0.7582,0.7416,0.7338,0.9948,0.7522,0.7503,0.9948,1.0
14,0.7691,0.7410,0.7574,0.8050,0.9972,0.7400,0.7733,0.9972,1.0
28,0.6924,0.7595,0.7249,0.7304,0.8864,0.7574,0.6936,0.8864,1.0


## Which model wins in which segment

The answerable version of "which model is best".

In [6]:
pd.DataFrame(r['train']['best_per_segment']).round(4)

,demand_class,n,best_model,best_wape,best_baseline,baseline_wape,improvement_pct,recommended_approach
0,Erratic,476,random_forest,0.7069,moving_average_7,0.7106,0.5226,Global ML model. The level is stable enough to...
1,Intermittent,12572,sba,0.9297,sba,0.9297,0.0000,Croston/SBA or a global ML model with a count-...
2,Lumpy,2884,sba,0.7974,sba,0.7974,0.0000,Hardest class: rare demand AND variable size. ...
3,Smooth,868,lightgbm,0.5175,moving_average_28,0.5523,6.3022,Classical exponential smoothing or a global ML...


## What drives forecast error

In [7]:
pd.DataFrame(r['train']['error_drivers']).round(4)

,characteristic,spearman_vs_wape,n
0,zero_share,0.5722,412
1,mean_demand,-0.5701,412
2,cv_demand,0.5460,412
3,total_demand,-0.5012,412
4,weekly_strength,-0.3435,412
5,snap_share,0.0508,412
6,history_days,-0.0308,412
7,price_cv,-0.0226,412


In [8]:
pd.DataFrame(r['train']['promo_errors']).round(4)

,condition,n,mean_actual,wape,mae,bias
0,snap_day,6000,1.4682,0.8033,1.1793,-0.0169
1,non_snap_day,10800,1.4764,0.7619,1.1248,-0.0868


## Uncertainty: are the intervals honest?

An unvalidated P90 is an assumption dressed as a measurement.

In [9]:
pd.DataFrame(r['train']['interval_report']).round(4)

,n,nominal_coverage,empirical_coverage,mean_interval_width,median_interval_width,coverage_gap,pinball_q10,exceedance_above_q10,pinball_q50,exceedance_above_q50,pinball_q90,exceedance_above_q90
0,16800.0,0.8,0.7504,2.9568,1.9069,-0.0496,0.1474,0.4423,0.5291,0.3648,0.344,0.1129


In [10]:
pd.DataFrame(r['train']['interval_by_horizon']).round(4).head(28)

,horizon,n,nominal_coverage,empirical_coverage,mean_interval_width,median_interval_width,coverage_gap,pinball_q10,exceedance_above_q10,pinball_q50,exceedance_above_q50,pinball_q90,exceedance_above_q90
0,1,600.0,0.8,0.7600,2.7238,1.7738,-0.0400,0.1302,0.4583,0.4280,0.3700,0.2756,0.1117
1,2,600.0,0.8,0.7583,2.6603,1.7209,-0.0417,0.1274,0.4267,0.4275,0.3467,0.3062,0.1067
2,3,600.0,0.8,0.7500,2.6430,1.7111,-0.0500,0.1198,0.4050,0.4298,0.3367,0.2704,0.0983
3,4,600.0,0.8,0.7367,2.6513,1.7260,-0.0633,0.1189,0.4183,0.4421,0.3400,0.3123,0.1117
4,5,600.0,0.8,0.7650,2.9057,1.8723,-0.0350,0.1302,0.4417,0.4935,0.3717,0.3183,0.0850
5,6,600.0,0.8,0.7733,3.3910,2.1926,-0.0267,0.1599,0.5217,0.5485,0.4050,0.3141,0.1083
6,7,600.0,0.8,0.7883,3.4601,2.2174,-0.0117,0.1602,0.4883,0.5657,0.3883,0.3065,0.0817
7,8,600.0,0.8,0.6717,2.4137,1.3521,-0.1283,0.1323,0.4483,0.5065,0.4033,0.3313,0.1967
8,9,600.0,0.8,0.7333,2.6632,1.7270,-0.0667,0.1139,0.4083,0.4688,0.3333,0.3178,0.1083
9,10,600.0,0.8,0.7133,2.6690,1.7067,-0.0867,0.1436,0.4450,0.5444,0.3817,0.3577,0.1500
